# Lab4.3 — Compressão JPEG com Perdas (v2)
**Parte B | 75–90 min.** Gere três versões JPEG para duas imagens: uma fotografia do grupo e uma imagem do banco anterior.
**Objetivos:** experimentar parâmetros JPEG, medir redução de tamanho e preparar imagens para avaliação externa por pares no Lab4.5.
O parâmetro FFmpeg `-q:v` **não** usa a escala 0–100: valores menores representam melhor qualidade.

In [ ]:
from pathlib import Path
import os
from google.colab import drive
drive.mount('/content/drive')
GRUPO = "G4"  # ALTERAR: G1, G2, ..., G6
RAIZ = Path('/content/drive/MyDrive/Lab4_CSM')
PASTA = RAIZ / GRUPO
PASTA.mkdir(parents=True, exist_ok=True)
print("Pasta:", PASTA)

## 1. Escolha duas imagens
Os arquivos devem estar em `imagens/originais`. Coloque a imagem do banco anterior nessa pasta e edite a lista abaixo.

In [ ]:
import subprocess, pandas as pd, shutil
ORIG=PASTA/'imagens'/'originais'
SAIDA=PASTA/'imagens'/'jpeg'
SAIDA.mkdir(parents=True,exist_ok=True)
ARQUIVOS=sorted(p.name for p in ORIG.glob('G4_*.png')) # ALTERAR para os nomes reais
assert shutil.which('ffmpeg'), 'FFmpeg indisponível'
def executar(args):
    r=subprocess.run(args,capture_output=True,text=True)
    if r.returncode: raise RuntimeError(r.stderr[-1200:])

## 2. Compressão e medidas
Execute a célula para gerar três níveis de qualidade para cada imagem. O CSV inclui a redução relativa ao arquivo original.

In [ ]:
linhas=[]
for nome in ARQUIVOS:
    origem=ORIG/nome
    assert origem.is_file(),f'Imagem ausente: {origem}'
    for q in (2,20,31):
        destino=SAIDA/f'{origem.stem}_q{q}.jpg'
        executar(['ffmpeg','-y','-loglevel','error','-i',str(origem),'-frames:v','1','-q:v',str(q),str(destino)])
        linhas.append({'grupo':GRUPO,'imagem':origem.name,'q_ffmpeg':q,
                       'bytes_original':origem.stat().st_size,'bytes_jpeg':destino.stat().st_size,
                       'reducao_pct':round(100*(1-destino.stat().st_size/origem.stat().st_size),2),
                       'arquivo_jpeg':destino.name})
medidas=pd.DataFrame(linhas)
display(medidas)
medidas.to_csv(PASTA/'imagens'/'medidas_imagem.csv',index=False)

## 3. Preparação da avaliação
Abra **Lab4.5** para criar os pares anônimos, compartilhar o pacote com o grupo parceiro e recolher as avaliações.